In [ ]:
# ==========================================
# Install (لو محتاج)
# ==========================================
!pip install -q transformers datasets evaluate torchvision

# ==========================================
# ==========================================
# Install
# ==========================================
!pip install -q transformers datasets evaluate torchvision

# ==========================================
# Imports
# ==========================================
import torch
import numpy as np
import evaluate
from datasets import load_dataset, Dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)
from torchvision.transforms import Compose, Normalize, ToTensor, Resize

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# Load small subset safely (no disk explosion)
# ==========================================

streamed = load_dataset(
    "ideepankarsharma2003/Midjourney_v6_Classification_small_shuffled",
    split="train",
    streaming=True
)

small_data = list(streamed.take(50))   # 👈 50 صورة فقط
dataset = Dataset.from_list(small_data)

dataset = dataset.train_test_split(test_size=0.2)

# ==========================================
# Extract labels correctly (FIXED)
# ==========================================

unique_labels = sorted(set(dataset["train"]["label"]))

label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

# Remap labels to 0..N
def remap_labels(example):
    example["label"] = label2id[example["label"]]
    return example

dataset = dataset.map(remap_labels)

# ==========================================
# Load Model
# ==========================================

checkpoint = "google/vit-base-patch16-224"

image_processor = AutoImageProcessor.from_pretrained(checkpoint)

model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(device)

# ==========================================
# Transforms
# ==========================================

def preprocess_images(examples):
    # The image_processor handles all the necessary transformations
    # including resizing, normalization, and converting to tensors.
    # It takes a list of PIL Images and returns a dictionary with 'pixel_values'.
    inputs = image_processor(examples["image"], return_tensors="pt")
    inputs["labels"] = examples["label"]
    return inputs

dataset = dataset.map(preprocess_images, batched=True, remove_columns=["image"])

# ==========================================
# Metric
# ==========================================

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy.compute(
            predictions=preds,
            references=labels
        )["accuracy"]
    }

# ==========================================
# Training Arguments
# ==========================================

training_args = TrainingArguments(
    output_dir="./vit-mini",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    num_train_epochs=1,
    logging_steps=5,
    save_strategy="no",
    report_to="none"
)

# ==========================================
# Trainer
# ==========================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics
)

# ==========================================
# Train
# ==========================================

trainer.train()

# ==========================================
# Evaluate
# ==========================================

metrics = trainer.evaluate()
print(metrics)